In [2]:
import json
import requests
import numpy as np
import random
import time

# api申请
https://dev.elsevier.com/index.jsp

In [3]:
API_KEY = "91e9b934ab2cfe7cd62394ef9e399f42"
#API_KEY = "ac006f61df89849ef5f37e59e6ff749b"
#search_term = "SCR"
#search_term = "Machine Learning"
#query = f'TITLE("{search_term}")'
#query = 'Selective AND Catalytic AND Reduction'

query = '("Selective AND Catalytic AND Reduction")'

base_url = 'https://api.elsevier.com/content/search/sciencedirect'
#base_url = 'https://api.elsevier.com/content/search?qs=title("%' + query + '%")/sciencedirect'
#base_url = f'https://api.elsevier.com/content/search/sciencedirect?query={query}&apiKey={API_KEY}'

user_agent_list = ["Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/68.0.3440.106 Safari/537.36",
                    "Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/67.0.3396.99 Safari/537.36",
                    "Mozilla/5.0 (Windows NT 10.0; WOW64) Gecko/20100101 Firefox/61.0",
                    "Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/64.0.3282.186 Safari/537.36",
                    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/62.0.3202.62 Safari/537.36",
                    "Mozilla/5.0 (Windows NT 6.1; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/45.0.2454.101 Safari/537.36",
                    "Mozilla/4.0 (compatible; MSIE 7.0; Windows NT 6.0)",
                    "Mozilla/5.0 (Macintosh; U; PPC Mac OS X 10.5; en-US; rv:1.9.2.15) Gecko/20110303 Firefox/3.6.15",
                    ]


data = {"qs": query,
        "date": 2021,
        "volume": 1,
        "display": {
            "show": 25,
            "offset": 0
        }}

headers = {'User-Agent': random.choice(user_agent_list),
            'x-els-apikey': API_KEY,
           'Content-Type': 'application/json',
           'Accept': 'application/json'}

In [4]:
def get_response(url, data, year,headers):
    """
    Return the response from Elsevier
    :param url: <str> base_url
    :param data: <dict> data parameters
    :param headers: <dict> headers
    :return: response
    """
    data["date"] = year
    response = requests.put(url, data=json.dumps(data), headers=headers)
    response = response.text.replace('false', 'False').replace('true', 'True')
    try:
        response = eval(response)
    except BaseException:
        print(response)
    return response

In [5]:
def get_doi(data,year):
    """
    Get DOIs
    :param data: <dict> data parameters
    :param volume: <int> the volume index
    :param year: <int> the year index
    :return: <list> of <str> list of DOIs
    """
    dois = []
    volume=1
    data['volume'] = volume
    data["date"] = year
    response = get_response(base_url, data,year, headers)
    if 'resultsFound' in response.keys():
        n = int(np.ceil(response['resultsFound']))
    else:
        n = 60
    for offset in range(n + 1):
        data["display"]["offset"] = offset
        response = get_response(base_url, data, year,headers)
        if 'results' in response.keys():
            results = response['results']
            for result in results:
                if 'doi' in result:
                    dois.append(result['doi'])
                    print(len(dois))
        
    return dois

In [6]:
def download_doi(doi):
    for doi2 in doi:
        
        doi2 = doi2.replace('/','')
    
        with open(str(doi2) + '.xml', 'w', encoding='utf-8') as f:
            request_url = 'https://api.elsevier.com/content/article/doi/{}?apiKey={}&httpAccept=text%2Fxml'.format(
                    doi, API_KEY)
            text = requests.post(request_url).text
            
            f.write(text)
        #time.sleep(1)
    return

def download_doi1(doi):
    
        
    doi3 = doi.replace('/','')
    
    with open(str(doi3) + '.xml', 'w', encoding='utf-8') as f:
        request_url = 'https://api.elsevier.com/content/article/doi/{}?apiKey={}&httpAccept=text%2Fxml'.format(
                doi, API_KEY)
        text = requests.post(request_url).text
            
        f.write(text)
        #time.sleep(1)
    return

In [7]:
response_2021 = get_response(base_url,data,2021,headers)
response_2021

{'service-error': {'status': {'statusCode': 'AUTHORIZATION_ERROR',
   'statusText': 'The requestor is not authorized to access the requested view or fields of the resource'}}}

In [8]:
response_2022 = get_response(base_url,data,2022,headers)
response_2022

{'service-error': {'status': {'statusCode': 'AUTHORIZATION_ERROR',
   'statusText': 'The requestor is not authorized to access the requested view or fields of the resource'}}}

In [10]:
dois_2022 = get_doi(data,2022)

In [11]:
doi = list(dois_2022)

In [12]:
for i in range(len(dois_2022)):
    download_doi1(doi[i])